<a href="https://colab.research.google.com/github/sromerar/omics-tutorials/blob/main/01_atlas_getting_started.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ATLAS: Getting Started

Annotated reproduction of the official ATLAS getting-started tutorial for multimodal single-cell trajectory inference.

## 1. Environment setup

ATLAS is installed from PyPI using the `atlas-smilies` package.

Because Google Colab comes with a preconfigured Python environment, installing ATLAS may update existing dependencies. I therefore verify the core package imports and record their versions before proceeding with the analysis.

### Environment check

Before installing ATLAS, I record the base Google Colab environment. This is useful for diagnosing dependency conflicts and makes the computational environment more transparent and reproducible.

In [2]:
!pip install "numpy==2.0.2" "pandas==2.2.3" "atlas-smilies==1.0.0"

### Verify the installation

Confirm that ATLAS and the core single-cell dependencies import successfully, and record their versions for reproducibility.

In [3]:
import atlas
import scanpy as sc
import muon as mu
import pandas as pd
import numpy as np

print("ATLAS imported successfully")
print("NumPy:", np.__version__)
print("pandas:", pd.__version__)
print("Scanpy:", sc.__version__)
print("muon:", mu.__version__)

ATLAS imported successfully
NumPy: 2.0.2
pandas: 2.2.3
Scanpy: 1.12
muon: 0.1.9


### Install ATAC fragment-file dependency

The official ATLAS workflow uses `pysam` when working with indexed scATAC-seq fragment files. I install and verify it separately because it is not included as a core `atlas-smilies` dependency.

In [6]:
!pip install pysam

  Using cached pysam-0.24.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (1.6 kB)
Using cached pysam-0.24.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (23.2 MB)


In [7]:
import pysam

print("pysam:", pysam.__version__)

pysam: 0.24.0


## 2. Download the tutorial data

The official ATLAS getting-started tutorial uses the 10x Genomics E18 mouse brain multiome dataset, which contains paired gene-expression and chromatin-accessibility measurements from the same cells.

I also download the cell-type annotations used in the tutorial from the MultiVelo repository.

In [8]:
!mkdir -p data && cd data \
    && wget -q https://raw.githubusercontent.com/welch-lab/MultiVelo/main/Examples/cell_annotations.tsv \
    && curl -L -O https://cf.10xgenomics.com/samples/cell-arc/1.0.0/e18_mouse_brain_fresh_5k/e18_mouse_brain_fresh_5k_filtered_feature_bc_matrix.tar.gz \
    && tar -xzf e18_mouse_brain_fresh_5k_filtered_feature_bc_matrix.tar.gz \
    && curl -L -O https://cf.10xgenomics.com/samples/cell-arc/1.0.0/e18_mouse_brain_fresh_5k/e18_mouse_brain_fresh_5k_atac_fragments.tsv.gz \
    && curl -L -O https://cf.10xgenomics.com/samples/cell-arc/1.0.0/e18_mouse_brain_fresh_5k/e18_mouse_brain_fresh_5k_atac_fragments.tsv.gz.tbi

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  194M  100  194M    0     0  17.8M      0  0:00:10  0:00:10 --:--:-- 22.4M
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  987M  100  987M    0     0  21.7M      0  0:00:45  0:00:45 --:--:-- 25.3M
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  709k  100  709k    0     0   399k      0  0:00:01  0:00:01 --:--:--  398k


### Verify downloaded files

I inspect the downloaded data directory to confirm that the 10x feature-barcode matrix, ATAC fragments file, fragment index, and cell-type annotation file are present before proceeding.

In [9]:
!ls -lh data

total 1.2G
-rw-r--r-- 1 root root 133K Aug 19 10:16 cell_annotations.tsv
-rw-r--r-- 1 root root 988M Aug 19 10:17 e18_mouse_brain_fresh_5k_atac_fragments.tsv.gz
-rw-r--r-- 1 root root 710K Aug 19 10:17 e18_mouse_brain_fresh_5k_atac_fragments.tsv.gz.tbi
-rw-r--r-- 1 root root 195M Aug 19 10:16 e18_mouse_brain_fresh_5k_filtered_feature_bc_matrix.tar.gz
drwxr-xr-x 2 5233 5000 4.0K Sep  5  2020 filtered_feature_bc_matrix


## 3. Load packages and define file paths

I import the libraries used throughout the ATLAS workflow, set a random seed for reproducibility, and define paths to the downloaded 10x multiome data and cell-type annotations.

In [10]:
import os
import atlas
import scanpy as sc
import muon as mu
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from muon import MuData
from anndata import AnnData
from scipy.stats import median_abs_deviation

seed = 42
np.random.seed(seed)

data_path = os.path.join(os.getcwd(), "data")
annotation_path = os.path.join(data_path, "cell_annotations.tsv")
ffbcm = os.path.join(data_path, "filtered_feature_bc_matrix")

print("Data path:", data_path)
print("Annotations:", annotation_path)
print("10x matrix:", ffbcm)

Data path: /content/data
Annotations: /content/data/cell_annotations.tsv
10x matrix: /content/data/filtered_feature_bc_matrix


## 4. Load the multimodal 10x data

The 10x feature-barcode matrix contains both gene-expression and chromatin-accessibility features measured from the same cells. I load the complete matrix and then separate it into RNA and ATAC `AnnData` objects using the feature annotations provided by 10x Genomics.

In [11]:
data = sc.read_10x_mtx(
    ffbcm,
    var_names="gene_symbols",
    gex_only=False
)

rna = data[:, data.var["feature_types"] == "Gene Expression"].copy()
atac = data[:, data.var["feature_types"] != "Gene Expression"].copy()

print("Combined matrix:", data.shape)
print("RNA:", rna.shape)
print("ATAC:", atac.shape)
print("\nFeature types:")
print(data.var["feature_types"].value_counts())

Combined matrix: (4881, 176722)
RNA: (4881, 32285)
ATAC: (4881, 144437)

Feature types:
feature_types
Peaks              144437
Gene Expression     32285
Name: count, dtype: int64


### Inspect feature metadata

ATLAS requires genomic coordinates for genes when calculating gene-activity scores from scATAC-seq fragments. The 10x `features.tsv.gz` file contains the feature annotations used to construct these coordinates.

In [12]:
features_path = os.path.join(ffbcm, "features.tsv.gz")

features = pd.read_csv(
    features_path,
    sep="\t",
    header=None
)

print("Feature metadata shape:", features.shape)
features.head()

Feature metadata shape: (176722, 6)


,0,1,2,3,4,5
0,ENSMUSG00000051951,Xkr4,Gene Expression,chr1,3671497,3671498
1,ENSMUSG00000089699,Gm1992,Gene Expression,chr1,3466586,3466587
2,ENSMUSG00000102331,Gm19938,Gene Expression,chr1,3658903,3658904
3,ENSMUSG00000102343,Gm37381,Gene Expression,chr1,3985983,3986215
4,ENSMUSG00000025900,Rp1,Gene Expression,chr1,4360313,4409241


### Construct gene genomic coordinates

ATLAS computes gene-activity scores by linking scATAC-seq fragments to genes. This requires genomic coordinates for each gene. I extract these coordinates from the 10x feature metadata and retain genes located on the standard mouse chromosomes.

In [13]:
features.columns = [
    "gene_id",
    "symbol",
    "feature_type",
    "Chromosome",
    "Start",
    "End",
]

gene_coordinates = (
    features.loc[
        features["feature_type"] == "Gene Expression",
        ["symbol", "Chromosome", "Start", "End"]
    ]
    .set_index("symbol")
)

standard_chromosomes = (
    [f"chr{i}" for i in range(1, 20)]
    + ["chrX", "chrY"]
)

gene_coordinates = gene_coordinates[
    gene_coordinates["Chromosome"].isin(standard_chromosomes)
]

print("Genes with genomic coordinates:", gene_coordinates.shape[0])
gene_coordinates.head()

Genes with genomic coordinates: 32195


,Chromosome,Start,End
symbol,,,
Xkr4,chr1,3671497,3671498
Gm1992,chr1,3466586,3466587
Gm19938,chr1,3658903,3658904
Gm37381,chr1,3985983,3986215
Rp1,chr1,4360313,4409241


### Link the ATAC fragments file

The fragment file contains the genomic coordinates of sequenced chromatin-accessibility fragments for each cell. I register this indexed file with the ATAC `AnnData` object so that ATLAS/muon can calculate ATAC-specific quality-control metrics and later compute gene-activity scores.

In [14]:
fragment_file_path = os.path.join(
    data_path,
    "e18_mouse_brain_fresh_5k_atac_fragments.tsv.gz"
)

mu.atac.tl.locate_fragments(atac, fragment_file_path)

print("Fragment file linked successfully.")

Fragment file linked successfully.


### Inspect the ATAC fragment file

Each row of the fragment file records an accessible chromatin fragment, including its chromosome, start and end coordinates, cell barcode, and read support. I inspect a few rows to confirm the expected structure before proceeding with ATAC quality control.

In [15]:
!zcat data/e18_mouse_brain_fresh_5k_atac_fragments.tsv.gz | head -n 3

chr1	3000076	3000232	GGCCATCAGGCGCTTA-1	1
chr1	3000234	3000479	GAACCAAAGGGACTAA-1	2
chr1	3000382	3000630	GTCCGTAAGCCTGTTC-1	3
